In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)

df = pd.read_csv("HousePricePrediction.csv")

print(df.info())
print(df.isnull().sum()[df.isnull().sum() > 0])
print(df.describe().T)

In [ ]:
target_col = 'SalePrice'

df = df.drop(columns=['Id'], errors='ignore')
df = df.dropna(subset=[target_col]).drop_duplicates().reset_index(drop=True)

num_cols = df.select_dtypes(include=[np.number]).columns.drop(target_col).tolist()
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(df.shape)

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.histplot(df[target_col], kde=True, color='teal')
plt.title(f"{target_col} Distribution (Skewness: {df[target_col].skew():.2f})")

plt.subplot(1, 2, 2)
sns.boxplot(x=df[target_col], color='lightcoral')
plt.title(f"{target_col} Boxplot")
plt.tight_layout()
plt.show()

## As residuals are hetergenous,but models like lr,log reg assumes homoscadisticity..so make log tranformation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df[target_col], kde=True, ax=axes[0], color='teal')
axes[0].set_title(f"Original SalePrice (Skew: {df[target_col].skew():.2f})")

log_price = np.log1p(df[target_col])
sns.histplot(log_price, kde=True, ax=axes[1], color='coral')
axes[1].set_title(f"Log-Transformed SalePrice (Skew: {log_price.skew():.2f})")

plt.tight_layout()
plt.show()

In [ ]:
df[num_cols].hist(figsize=(14, 10), bins=25, edgecolor="black", color="#4C72B0")
plt.suptitle("Numerical Features Distribution", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
df[target_col] = np.log1p(df[target_col])

In [ ]:
df['MSSubClass'] = df['MSSubClass'].astype(str)
df['LotArea'] = np.log1p(df['LotArea'])
df['Has_BsmtFin2'] = (df['BsmtFinSF2'] > 0).astype(int)
df['HouseAge'] = df['YearBuilt'].max() - df['YearBuilt']
df['IsRemodeled'] = (df['YearRemodAdd'] != df['YearBuilt']).astype(int)
df = df.drop(columns=['YearBuilt'])

num_cols = df.select_dtypes(include=[np.number]).columns.drop(target_col).tolist()
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

df[num_cols].hist(figsize=(14, 10), bins=25, edgecolor="black", color="#4C72B0")
plt.suptitle("Numerical Features Distribution", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))
for i, col in enumerate(num_cols, 1):
    plt.subplot((len(num_cols) + 3) // 4, 4, i)
    sns.boxplot(y=df[col], color="skyblue")
    plt.title(col)
plt.suptitle("Outlier Detection via Boxplots", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
corr_matrix = df.select_dtypes(include=[np.number]).corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Matrix Heatmap", fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))
for i, col in enumerate(cat_cols, 1):
    plt.subplot((len(cat_cols) + 2) // 3, 3, i)
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, hue=col, palette="viridis", legend=False)
    plt.title(f"Distribution of {col}")
    plt.xticks(rotation=45)
plt.suptitle("Categorical Features Distribution", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import train_test_split, cross_val_score, KFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

In [ ]:
X = df.drop(columns=[target_col])
y = df[target_col]

num_features = X.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape)
print(num_features)
print(cat_features)

In [ ]:
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(min_frequency=0.01, handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_features),
        ('cat', cat_pipeline, cat_features)
    ],
    remainder='drop'
)

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.0005, max_iter=5000),
    "Support Vector Regressor": SVR(C=1000, epsilon=0.1),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=150, random_state=42),
    "Gradient Boosting Regressor": GradientBoostingRegressor(n_estimators=150, learning_rate=0.08, random_state=42),
    "XGBoost Regressor": XGBRegressor(n_estimators=150, learning_rate=0.08, random_state=42, verbosity=0)
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
results = []

for model_name, model in models.items():
    full_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', model)])

    cv_scores = cross_val_score(full_pipeline, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error')

    full_pipeline.fit(X_train, y_train)
    y_pred = full_pipeline.predict(X_test)

    results.append({
        "Model": model_name,
        "CV RMSE": round(-cv_scores.mean(), 4),
        "Test MAE": round(mean_absolute_error(y_test, y_pred), 4),
        "Test RMSE": round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
        "Test R2": round(r2_score(y_test, y_pred), 4)
    })

results_df = pd.DataFrame(results).sort_values(by="CV RMSE").reset_index(drop=True)
print(results_df.to_string(index=False))

In [ ]:
gb_param_grid = {
    'regressor__n_estimators': [100, 200, 300],
    'regressor__learning_rate': [0.03, 0.05, 0.1],
    'regressor__max_depth': [3, 4, 5],
    'regressor__subsample': [0.8, 1.0],
    'regressor__min_samples_split': [2, 5]
}

xgb_param_grid = {
    'regressor__n_estimators': [100, 200, 300],
    'regressor__learning_rate': [0.03, 0.05, 0.1],
    'regressor__max_depth': [3, 4, 6],
    'regressor__colsample_bytree': [0.7, 0.8, 1.0],
    'regressor__subsample': [0.7, 0.8, 1.0]
}

rf_param_grid = {
    'regressor__n_estimators': [150, 250, 350],
    'regressor__max_depth': [10, 15, 20, None],
    'regressor__min_samples_split': [2, 5, 10],
    'regressor__min_samples_leaf': [1, 2, 4],
    'regressor__max_features': ['sqrt', 'log2', 1.0]
}

models_to_tune = [
    ("Tuned Gradient Boosting", GradientBoostingRegressor(random_state=42), gb_param_grid),
    ("Tuned XGBoost", XGBRegressor(random_state=42, verbosity=0), xgb_param_grid),
    ("Tuned Random Forest", RandomForestRegressor(random_state=42), rf_param_grid)
]

In [ ]:
tuned_results = []
best_fitted_models = {}

for name, model, param_grid in models_to_tune:
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', model)])

    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_grid,
        n_iter=20,
        cv=cv,
        scoring='neg_root_mean_squared_error',
        random_state=42,
        n_jobs=-1
    )
    search.fit(X_train, y_train)

    best_pipe = search.best_estimator_
    best_fitted_models[name] = best_pipe
    y_pred = best_pipe.predict(X_test)

    tuned_results.append({
        "Model": name,
        "Best CV RMSE": round(-search.best_score_, 4),
        "Test MAE": round(mean_absolute_error(y_test, y_pred), 4),
        "Test RMSE": round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
        "Test R2": round(r2_score(y_test, y_pred), 4),
        "Best Params": search.best_params_
    })
    print(name, "done")

In [ ]:
tuned_df = pd.DataFrame(tuned_results).sort_values(by="Best CV RMSE").reset_index(drop=True)
print(tuned_df.drop(columns=['Best Params']).to_string(index=False))

for res in tuned_results:
    print(res['Model'])
    for k, v in res['Best Params'].items():
        print(" ", k.replace('regressor__', ''), v)

In [ ]:
best_name = tuned_df.loc[0, 'Model']
best_model = best_fitted_models[best_name]

y_pred_log = best_model.predict(X_test)
y_test_dollars = np.expm1(y_test)
y_pred_dollars = np.expm1(y_pred_log)
residuals_dollars = y_test_dollars - y_pred_dollars

print(best_name)
print("R2 (log scale):", round(r2_score(y_test, y_pred_log), 4))
print("RMSE (log scale):", round(np.sqrt(mean_squared_error(y_test, y_pred_log)), 4))
print("MAE ($):", round(mean_absolute_error(y_test_dollars, y_pred_dollars), 2))
print("RMSE ($):", round(np.sqrt(mean_squared_error(y_test_dollars, y_pred_dollars)), 2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.scatterplot(x=y_test_dollars, y=y_pred_dollars, alpha=0.7, color='teal', ax=axes[0])
axes[0].plot([y_test_dollars.min(), y_test_dollars.max()],
             [y_test_dollars.min(), y_test_dollars.max()], 'r--', lw=2, label="Perfect 1:1 Fit")
axes[0].set_title("Actual vs Predicted Prices ($)")
axes[0].set_xlabel("Actual Price ($)")
axes[0].set_ylabel("Predicted Price ($)")
axes[0].legend()

sns.scatterplot(x=y_pred_dollars, y=residuals_dollars, alpha=0.7, color='purple', ax=axes[1])
axes[1].axhline(0, color='red', linestyle='--', lw=2)
axes[1].set_title("Residuals vs Predicted")
axes[1].set_xlabel("Predicted Price ($)")
axes[1].set_ylabel("Residual Error ($)")

sns.histplot(residuals_dollars, kde=True, color='royalblue', ax=axes[2])
axes[2].axvline(0, color='red', linestyle='--', lw=2)
axes[2].set_title("Residual Distribution (Errors)")
axes[2].set_xlabel("Error ($)")

plt.tight_layout()
plt.show()

In [ ]:
import pickle
from pathlib import Path

final_model = clone(best_model)
final_model.fit(X, y)

file_path = Path.cwd() / "house_price_champion_pipeline.pkl"
with open(file_path, "wb") as f:
    pickle.dump(final_model, f)

print("Saved:", file_path)